In [23]:
import sys
import os
import numpy as np 
import pandas as pd 
from torch.utils.data import DataLoader  
import yaml
from sr_model import paired_sr, single_sr
from dataset import omic_data,single_data
import torch
import muon as mu 
import json 
import scanpy as sc 

'''
train a paired model without pretraining weights
'''
def load_data():
    mdata = mu.read_h5mu('/home/rsun@ZHANGroup.local/multi_pretrain/evaluation/notebook/eval_data/M_rna_1/mdata.h5mu')
    
    rna = mdata['rna_count']
    gadata = mdata['ga_count'] 

    mutual_gene = np.load('/home/rsun@ZHANGroup.local/atac_pretrain/src/mutual_gene.npy', allow_pickle=True)
    gadata = gadata[:,mutual_gene]
    
    # lo1p transform 

    sc.pp.normalize_total(gadata, target_sum= 1e4)
    sc.pp.log1p(gadata)

    sc.pp.normalize_total(rna, target_sum= 1e4)
    sc.pp.log1p(rna)

    # feature selection

    sc.pp.highly_variable_genes(rna, n_top_genes= 5000)
    rna = rna[:,rna.var['highly_variable']]

    sc.pp.highly_variable_genes(gadata, n_top_genes= 10000)
    gadata = gadata[:,gadata.var['highly_variable']] 

    print(rna.shape, gadata.shape)

    #train_idx, test_idx = train_test_split(np.arange(N), test_size = 0.1, random_state = 42)
    train_idx = np.load('/home/rsun@ZHANGroup.local/multi_pretrain/evaluation/sr_result/train_test_split/train_id_1.npy', allow_pickle = True)
    test_idx = np.load('/home/rsun@ZHANGroup.local/multi_pretrain/evaluation/sr_result/train_test_split/test_id_1.npy', allow_pickle = True)

    rna_train, rna_test = rna[train_idx,:], rna[test_idx,:]
    gadata_train, gadata_test = gadata[train_idx,:], gadata[test_idx,:]
    return rna_train, rna_test, gadata_train, gadata_test

def process_data(rna_train, rna_test, gadata_train, gadata_test):

    train_rna = rna_train.X.toarray().astype(np.float32)
    train_ga = gadata_train.X.toarray().astype(np.float32)
    train_rna = torch.from_numpy(train_rna)
    train_ga = torch.from_numpy(train_ga)

    test_rna = rna_test.X.toarray().astype(np.float32)
    test_ga = gadata_test.X.toarray().astype(np.float32)
    test_rna = torch.from_numpy(test_rna)
    test_ga = torch.from_numpy(test_ga)
    return train_rna, train_ga, test_rna, test_ga

def set_rna_config(N, B=1024):
    # N is the dataset size , B is the batch size 

    if N >= 100000:
        print('Large dataset, please use pair_train.py')
        return None 
    steps = int(40*N/B) # at least 1000 steps 


    config = {
        'omic': {'model_type': 'rna'},

        'network': {
            'feature_num': 5000,
            'hidden_dims': [512, 256, 128],
            'dropout': 0.1,
            'layernorm_eps': 1e-8,
            'activation': 'leaky_relu',
            'input_dropout': 0.2,
            'vae_weight': 0,
            'ce_weights': None,
            'class_dict': None
        },
        'optimizer': {
            'learning_rate': 1e-4,
            'weight_decay': 0.01,
            'warmup_steps': 100,
            'anneal_steps': steps,
            'min_lr': 1e-6
        },
        'training': {
            'device': 'cuda',
            'training_steps': steps,
            'eval_steps': 100000,
            'save_steps': 100000, # set large, do not save checkpoint during training
            'log_dir': 'logs',
            'save_dir': 'saved_models',
            'run_name': 'tiny_rna'
        }
    }
    return config

def set_ga_config(N, B=1024):
    # N is the dataset size , B is the batch size 

    if N >= 100000:
        print('Large dataset, please use pair_train.py')
        return None 
    steps = int(60*N/B) # at least 1000 steps 


    config = {
        'omic': {'model_type': 'ga'},

        'network': {
            'feature_num': 10000,
            'hidden_dims': [512, 256, 128],
            'dropout': 0.1,
            'layernorm_eps': 1e-8,
            'activation': 'leaky_relu',
            'input_dropout': 0.2,
            'vae_weight': 0,
            'ce_weights': None,
            'class_dict': None
        },
        'optimizer': {
            'learning_rate': 1e-4,
            'weight_decay': 0.01,
            'warmup_steps': 100,
            'anneal_steps': steps,
            'min_lr': 1e-6
        },
        'training': {
            'device': 'cuda',
            'training_steps': steps,
            'eval_steps': 100000,
            'save_steps': 100000, # set large, do not save checkpoint during training
            'log_dir': 'logs',
            'save_dir': 'saved_models',
            'run_name': 'tiny_ga'
        }
    }
    return config
def rna_train(rna_data):
    print(1)
    rna_dataset = single_data(rna_data)
    print(rna_dataset)
    rna_loader = DataLoader(rna_dataset, batch_size = 1024, shuffle= True)
    for batch in rna_loader:
        print(batch)
        break

    '''
    ini config
    '''
    config = set_rna_config(N = rna_data.shape[0], B = 1024)
    print(config)

    '''
    set rna model 
    '''
    rna_model = single_sr(config)
    #rna_model.set_optimizer()
    rna_model.train_model(train_loader = rna_loader,save_config= False)
    return rna_model, config

def ga_train(ga_data):
    ga_dataset = single_data(ga_data)
    ga_loader = DataLoader(ga_dataset, batch_size = 1024, shuffle= True)

    '''
    ini config
    '''
    config = set_ga_config(N = ga_data.shape[0], B = 1024)

    '''
    set rna model 
    '''
    ga_model = single_sr(config)
    ga_model.set_optimizer()
    ga_model.train_model(train_loader = ga_loader)
    return ga_model, config 


In [3]:
rna_train, rna_test, gadata_train, gadata_test = load_data()
train_rna, train_ga, test_rna, test_ga = process_data(rna_train, rna_test, gadata_train, gadata_test)
print(train_rna.shape, train_ga.shape)
print(test_rna.shape, test_ga.shape)

combined_rna = torch.cat((train_rna, test_rna), dim=0)
combined_ga = torch.cat((train_ga, test_ga), dim=0)
print(combined_rna.shape, combined_ga.shape)

#rna_dataset = single_data(combined_rna)
#ga_dataset = single_data(combined_ga)

"""
prepare  multiomic data
"""
traindata = omic_data(train_rna, train_ga)
testdata = omic_data(test_rna, test_ga) 
print('dataset over')

'''
prepare dataloader 
'''
batchsize = 2048

train_loader = DataLoader(traindata, batch_size= batchsize, shuffle=True)
test_loader = DataLoader(testdata, batch_size= batchsize, shuffle=True)
print('dataloader over')

/home/rsun@ZHANGroup.local/anaconda3/envs/snapatac/lib/python3.10/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/rsun@ZHANGroup.local/anaconda3/envs/snapatac/lib/python3.10/site-packages/mudata/_core/mudata.py:931: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  warnings.warn(
/home/rsun@ZHANGroup.local/anaconda3/envs/snapatac/lib/python3.10/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. U

(71002, 5000) (71002, 10000)
torch.Size([63901, 5000]) torch.Size([63901, 10000])
torch.Size([7101, 5000]) torch.Size([7101, 10000])
torch.Size([71002, 5000]) torch.Size([71002, 10000])
dataset over
dataloader over


In [24]:
all_res = {}
for step_key in range(25,2025,25):
    checkpoint_path = checkpoint_path = f'/home/rsun@ZHANGroup.local/sr_project/saved_models/paired_config_scratch_2025-02-25-04-58/checkpoint_{step_key}.pth'

    with open('/home/rsun@ZHANGroup.local/sr_project/configs/paired_configs/config_scratch.yaml', 'r') as f:
        config = yaml.safe_load(f)

    ga_config = set_ga_config(N = test_ga.shape[0])
    rna_config = set_rna_config(N = test_rna.shape[0])


    paired_model = paired_sr(config,
                            rna_config = rna_config,
                            ga_config = ga_config,
                            sr_rna = None,
                            sr_ga = None)


    paired_model.load_checkpoint(checkpoint_path) 
    paired_model.model.to('cuda')

    rna_embed_list = []
    ga_embed_list = []

    eval_dic = {}
    eval_count = 0
    paired_model.model.eval()

    device = 'cuda'

    with torch.no_grad():
        for batch in test_loader:
            rna = batch['rna'].to(device)
            ga = batch['ga'].to(device)
            N = rna.shape[0] 
            eval_count += N 

            outputs = paired_model.model(rna, ga)
            for key in outputs:
                if 'loss' in key:
                    if key not in eval_dic:
                        eval_dic[key] = 0
                    if type(outputs[key]) == int:
                        eval_dic[key] += outputs[key] * N
                    else:
                        eval_dic[key] += outputs[key].item() * N 
            rna_embed = outputs['rna_embed'].detach().cpu().numpy() 
            ga_embed = outputs['ga_embed'].detach().cpu().numpy()
            rna_embed_list.append(rna_embed)
            ga_embed_list.append(ga_embed)

    for key in eval_dic:
        eval_dic[key] = eval_dic[key] / eval_count

    rna_embed = np.concatenate(rna_embed_list)
    ga_embed = np.concatenate(ga_embed_list) 

    eval_res = {}
    for k in [1,2,5,10,15,20,30,50,100]:
        tmp_s = calculate_hit_rate(rna_embed, ga_embed, K = k, metric = 'cosine')
        eval_res[f'top_{k}'] = tmp_s
    
    acc, matchscore, foscttm = matching_metrics( x=rna_embed, y=ga_embed, metric='cosine')
    eval_res['acc'] = acc
    eval_res['matchscore'] = matchscore
    eval_res['foscttm'] = foscttm
    print(f'{step_key}: {eval_res}')

    all_res[step_key] = eval_res


Initialize rna model
Initialize ga model
Initialize multi model
Checkpoint loaded from /home/rsun@ZHANGroup.local/sr_project/saved_models/paired_config_scratch_2025-02-25-04-58/checkpoint_25.pth at step 25
25: {'top_1': 0.0, 'top_2': 0.0, 'top_5': 0.0, 'top_10': 0.0, 'top_15': 0.0, 'top_20': 0.0, 'top_30': 0.0, 'top_50': 0.0, 'top_100': 0.0, 'acc': 0.00014082524285186082, 'matchscore': 0.00014082524285186082, 'foscttm': 0.6348186731338501}
Initialize rna model
Initialize ga model
Initialize multi model


/home/rsun@ZHANGroup.local/sr_project/src/sr_model.py:431: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Checkpoint loaded from /home/rsun@ZHANGroup.local/sr_project/saved_models/paired_config_scratch_2025-02-25-04-58/checkpoint_50.pth at step 50
50: {'top_1': 0.0, 'top_2': 0.0, 'top_5': 0.0, 'top_10': 0.0, 'top_15': 0.0, 'top_20': 0.0, 'top_30': 0.0, 'top_50': 0.0, 'top_100': 0.0, 'acc': 0.00014082524285186082, 'matchscore': 0.00014082524285186082, 'foscttm': 0.6122217178344727}
Initialize rna model
Initialize ga model
Initialize multi model


/home/rsun@ZHANGroup.local/sr_project/src/sr_model.py:431: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Checkpoint loaded from /home/rsun@ZHANGroup.local/sr_project/saved_models/paired_config_scratch_2025-02-25-04-58/checkpoint_75.pth at step 75
75: {'top_1': 0.0, 'top_2': 0.0, 'top_5': 0.0, 'top_10': 0.0, 'top_15': 0.0, 'top_20': 0.0, 'top_30': 0.0, 'top_50': 0.0, 'top_100': 0.0, 'acc': 0.00014082524285186082, 'matchscore': 0.00014082524285186082, 'foscttm': 0.5700841844081879}
Initialize rna model
Initialize ga model
Initialize multi model


...

Checkpoint loaded from /home/rsun@ZHANGroup.local/sr_project/saved_models/paired_config_scratch_2025-02-25-04-58/checkpoint_1950.pth at step 1950
1950: {'top_1': 0.0, 'top_2': 0.03795240107027179, 'top_5': 0.11104069849316997, 'top_10': 0.19138149556400508, 'top_15': 0.24693705111956063, 'top_20': 0.29545134488100266, 'top_30': 0.3672018025630193, 'top_50': 0.4659907055344318, 'top_100': 0.6004788058019997, 'acc': 0.07639768719673157, 'matchscore': 0.07815800607204437, 'foscttm': 0.013300795573741198}
Initialize rna model
Initialize ga model
Initialize multi model


/home/rsun@ZHANGroup.local/sr_project/src/sr_model.py:431: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Checkpoint loaded from /home/rsun@ZHANGroup.local/sr_project/saved_models/paired_config_scratch_2025-02-25-04-58/checkpoint_1975.pth at step 1975
1975: {'top_1': 0.0, 'top_2': 0.038656527249683145, 'top_5': 0.11413885368257992, 'top_10': 0.19356428672018025, 'top_15': 0.2501056189269117, 'top_20': 0.29580340797070837, 'top_30': 0.3717786227291931, 'top_50': 0.4723278411491339, 'top_100': 0.6034361357555273, 'acc': 0.07858048379421234, 'matchscore': 0.07942543178796768, 'foscttm': 0.013147496618330479}
Initialize rna model
Initialize ga model
Initialize multi model


/home/rsun@ZHANGroup.local/sr_project/src/sr_model.py:431: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Checkpoint loaded from /home/rsun@ZHANGroup.local/sr_project/saved_models/paired_config_scratch_2025-02-25-04-58/checkpoint_2000.pth at step 2000
2000: {'top_1': 0.0, 'top_2': 0.03971271651880017, 'top_5': 0.11406844106463879, 'top_10': 0.19419800028165046, 'top_15': 0.25207717222926346, 'top_20': 0.2984086748345304, 'top_30': 0.36931418110125336, 'top_50': 0.47282072947472187, 'top_100': 0.6040698493169976, 'acc': 0.07780594378709793, 'matchscore': 0.07759470492601395, 'foscttm': 0.013276303187012672}


In [25]:
df = pd.DataFrame(all_res)
df

,25,50,75,100,125,150,175,200,225,250,...,1775,1800,1825,1850,1875,1900,1925,1950,1975,2000
top_1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
top_2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.035910,0.037178,0.036967,0.035277,0.038797,0.036967,0.039079,0.037952,0.038657,0.039713
top_5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.106675,0.109773,0.108295,0.106605,0.112449,0.109140,0.113646,0.111041,0.114139,0.114068
top_10,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.183847,0.185819,0.187157,0.184622,0.190889,0.189128,0.192719,0.191381,0.193564,0.194198
top_15,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.240037,0.241938,0.242431,0.242149,0.244684,0.247078,0.250106,0.246937,0.250106,0.252077
top_20,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.284608,0.286931,0.289114,0.287283,0.292072,0.292001,0.296155,0.295451,0.295803,0.298409
top_30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.356922,0.356499,0.360513,0.358189,0.366286,0.363611,0.367272,0.367202,0.371779,0.369314
top_50,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.456555,0.458597,0.462118,0.461203,0.466976,0.464442,0.469511,0.465991,0.472328,0.472821
top_100,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.590128,0.592522,0.594846,0.593578,0.599071,0.596184,0.601183,0.600479,0.603436,0.604070
acc,0.000141,0.000141,0.000141,0.000211,0.000282,0.000493,0.000775,0.000634,0.001127,0.001479,...,0.079425,0.081397,0.079425,0.077736,0.079707,0.077665,0.079566,0.076398,0.078580,0.077806


In [9]:
rna_embed = np.concatenate(rna_embed_list)
ga_embed = np.concatenate(ga_embed_list)    

In [14]:
for k in [1,2,5,10,15,20,30,50,100]:
    print(calculate_hit_rate(rna_embed, ga_embed, K = k, metric = 'cosine'))
    print(calculate_hit_rate(rna_embed, ga_embed, K = k, metric = 'euclidean'))

0.0
0.0
0.03443177017321504
0.015349950711167442
0.09456414589494437
0.05048584706379383
0.16237149697225742
0.09632446134347275
0.21067455287987608
0.13350232361639205
0.2470778763554429
0.16363892409519787
0.30354879594423323
0.20743557245458386
0.3799464864103647
0.2732713702295451
0.48408674834530346
0.3661456132939023


In [12]:
import torch
import scipy.spatial
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sklearn.neighbors import NearestNeighbors

def matching_metrics(similarity=None, x=None, y=None, metric='euclidean', **kwargs):
    """
    计算匹配指标。

    参数:
        similarity (torch.Tensor, optional): 预计算的相似性矩阵。
        x (np.ndarray, optional): 第一个模态的嵌入向量。
        y (np.ndarray, optional): 第二个模态的嵌入向量。
        metric (str): 距离度量方式，支持 'euclidean' 或 'cosine'。
        **kwargs: 其他参数传递给距离计算函数。

    返回:
        acc: 准确率。
        matchscore: 匹配分数。
        foscttm: FOSCTTM 指标。
    """
    if similarity is None:
        if x.shape != y.shape:
            raise ValueError("Shapes do not match!")
        
        if metric == 'euclidean':
            # 计算欧式距离矩阵并转换为相似性矩阵
            distance_matrix = scipy.spatial.distance_matrix(x, y, **kwargs)
            similarity = 1 - distance_matrix
        elif metric == 'cosine':
            # 计算余弦相似性矩阵
            similarity = cosine_similarity(x, y)
        else:
            raise ValueError("Unsupported metric. Choose 'euclidean' or 'cosine'.")
    
    if not isinstance(similarity, torch.Tensor):
        similarity = torch.from_numpy(similarity)

    with torch.no_grad():
        batch_size = similarity.shape[0]
        
        # 计算 acc_x 和 acc_y
        acc_x = (
            torch.sum(
                torch.argmax(similarity, dim=1)
                == torch.arange(batch_size).to(similarity.device)
            )
            / batch_size
        )
        acc_y = (
            torch.sum(
                torch.argmax(similarity, dim=0)
                == torch.arange(batch_size).to(similarity.device)
            )
            / batch_size
        )
        
        # 计算 foscttm_x 和 foscttm_y
        foscttm_x = (
            (similarity > torch.diag(similarity)).float().mean(axis=1).mean().item()
        )
        foscttm_y = (
            (similarity > torch.diag(similarity)).float().mean(axis=0).mean().item()
        )
        
        # 计算 matchscore
        X = similarity
        mx = torch.max(X, dim=1, keepdim=True).values
        hard_X = (mx == X).float()
        logits_row_sums = hard_X.clip(min=0).sum(dim=1)
        matchscore = hard_X.clip(min=0).diagonal().div(logits_row_sums).mean().item()

        # 计算平均值
        acc = (acc_x + acc_y) / 2
        foscttm = (foscttm_x + foscttm_y) / 2
        
        return acc.item(), matchscore, foscttm

def calculate_hit_rate(rna_embeddings, atac_embeddings, K, metric = 'euclidean'):
    N = rna_embeddings.shape[0]
    # 合并嵌入和生成标签
    combined = np.concatenate([rna_embeddings, atac_embeddings], axis=0)
    cell_ids = np.concatenate([np.arange(N), np.arange(N)])  # 细胞ID
    modalities = np.array(['RNA']*N + ['ATAC']*N)  # 模态标签
    
    # 计算K近邻
    if metric == 'euclidean':
        #print(f'metric used {metric}')
        nbrs = NearestNeighbors(n_neighbors=K, metric='euclidean').fit(combined)
    elif metric == 'cosine':
        #print(f'metric used {metric}')
        nbrs = NearestNeighbors(n_neighbors=K, metric='cosine').fit(combined)
    else:
        print(f'metric used not support')
    _, indices = nbrs.kneighbors(combined)
    
    hit_count = 0
    for i in range(2*N):
        current_cell = cell_ids[i]
        current_modality = modalities[i]
        # 检查每个邻居
        for neighbor_idx in indices[i]:
            # 排除自身（若K包含自身需处理）
            if neighbor_idx == i:
                continue
            neighbor_cell = cell_ids[neighbor_idx]
            neighbor_modality = modalities[neighbor_idx]
            # 命中条件：同一细胞且不同模态
            if neighbor_cell == current_cell and neighbor_modality != current_modality:
                hit_count += 1
                break  # 至少一个命中即停止
    
    hit_rate = hit_count / (2 * N)
    return hit_rate
